# 6. Train PointNet++

Trains the PointNet++ SSG classifier on labeled obstacle clusters.

**Prerequisites:** Notebook 5 completed — `DATASET_DIR/train.csv`, `val.csv`, `test.csv` exist.

**Hardware:** Runs on CPU (~30–60 min/100 epochs for 5k samples) or CUDA GPU (~3–5 min).
Install PyTorch with: `pip install torch` (CPU) or see pytorch.org for GPU builds.
Optional faster ball query: `pip install torch-cluster`

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('.').resolve()))

from config import DATASET_DIR, MODELS_DIR
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Config ──────────────────────────────────────────────────────────────────
N_POINTS    = 512
N_FEATURES  = 7       # x_c, y_c, z, r, g, b, h_ag
BATCH_SIZE  = 32
EPOCHS      = 100
LR          = 1e-3
LR_DECAY    = 0.7
DECAY_STEP  = 20      # epochs between LR decay steps
WEIGHT_DECAY = 1e-4
AUGMENT     = True
MODEL_OUT   = MODELS_DIR / "pointnet2_classifier.pt"
LOG_EVERY   = 10      # epochs between per-class F1 logs

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

from utils.cluster_dataset import ClusterDataset
from utils.augmentation import default_train_augmentation
from utils.pointnet2 import build_model
from utils.labels import Labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == 'cpu':
    print("  Running on CPU — expect ~30–60 min for 100 epochs.")
    print("  Install GPU PyTorch (pytorch.org) for ~10x speedup.")

## Load datasets

In [ ]:
train_df = pd.read_csv(DATASET_DIR / "train.csv")
val_df   = pd.read_csv(DATASET_DIR / "val.csv")
test_df  = pd.read_csv(DATASET_DIR / "test.csv")

# Load label map from notebook 5
with open(DATASET_DIR / "class_map.json") as f:
    label_map = {int(k): int(v) for k, v in json.load(f).items()}

n_classes = len(set(label_map.values()))

# Label names
custom_labels = {}
from config import CLUSTERS_DIR
custom_path = CLUSTERS_DIR / "custom_labels.json"
if custom_path.exists():
    with open(custom_path) as f:
        custom_labels = {int(k): v for k, v in json.load(f).items()}
label_names_all = dict(Labels.STR_DICT)
label_names_all.update(custom_labels)

# class_idx → label_name
idx_to_name = {}
for code, idx in label_map.items():
    if idx not in idx_to_name:
        idx_to_name[idx] = label_names_all.get(code, str(code))
class_names = [idx_to_name[i] for i in range(n_classes)]

print(f"Classes: {n_classes}")
for i, name in enumerate(class_names):
    n_train = int((train_df['final_label'].map(label_map) == i).sum())
    n_test  = int((test_df['final_label'].map(label_map) == i).sum())
    print(f"  {i:>2d}  {name:<22}  train: {n_train}  test: {n_test}")

In [ ]:
aug = default_train_augmentation() if AUGMENT else None

train_ds = ClusterDataset(train_df, n_points=N_POINTS, augment_fn=aug, label_map=label_map)
val_ds   = ClusterDataset(val_df,   n_points=N_POINTS, augment_fn=None, label_map=label_map)
test_ds  = ClusterDataset(test_df,  n_points=N_POINTS, augment_fn=None, label_map=label_map)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=(device.type=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_weights = train_ds.class_weights().to(device)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")
print(f"Steps per epoch: {len(train_loader)}")

## Build model

In [ ]:
model = build_model(n_classes=n_classes, n_features=N_FEATURES, device=str(device))
total_params = sum(p.numel() for p in model.parameters())
print(f"PointNet++ SSG  |  {total_params:,} parameters")

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=DECAY_STEP, gamma=LR_DECAY)

## Training loop

In [ ]:
def evaluate(loader, model):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for pts, lbl in loader:
            pts, lbl = pts.to(device), lbl.to(device)
            logits = model(pts)
            total_loss += criterion(logits, lbl).item() * len(lbl)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(lbl.cpu().numpy())
    acc  = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    loss = total_loss / len(loader.dataset)
    return loss, acc, np.array(all_preds), np.array(all_labels)


train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    ep_correct = 0

    for pts, lbl in train_loader:
        pts, lbl = pts.to(device), lbl.to(device)
        optimizer.zero_grad()
        logits = model(pts)
        loss   = criterion(logits, lbl)
        loss.backward()
        optimizer.step()
        ep_loss    += loss.item() * len(lbl)
        ep_correct += (logits.argmax(dim=1) == lbl).sum().item()

    scheduler.step()

    tr_loss = ep_loss / len(train_ds)
    tr_acc  = ep_correct / len(train_ds)
    vl_loss, vl_acc, _, _ = evaluate(val_loader, model)

    train_losses.append(tr_loss); val_losses.append(vl_loss)
    train_accs.append(tr_acc);   val_accs.append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), MODEL_OUT)

    if epoch % LOG_EVERY == 0 or epoch == 1:
        _, _, val_preds, val_true = evaluate(val_loader, model)
        mf1 = f1_score(val_true, val_preds, average='macro', zero_division=0)
        print(f"Epoch {epoch:>3d}/{EPOCHS}  "
              f"train_loss={tr_loss:.4f}  val_loss={vl_loss:.4f}  "
              f"val_acc={vl_acc:.3f}  val_macro_F1={mf1:.3f}  "
              f"lr={scheduler.get_last_lr()[0]:.2e}")

print(f"\nBest val accuracy: {best_val_acc:.3f}  →  {MODEL_OUT}")

## Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)
ax1.plot(epochs, train_losses, label='train'); ax1.plot(epochs, val_losses, label='val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()
ax2.plot(epochs, train_accs, label='train'); ax2.plot(epochs, val_accs, label='val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig(MODELS_DIR / "training_curves.png", dpi=120)
plt.show()

## Final evaluation on test set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(MODEL_OUT, map_location=device))
_, test_acc, test_preds, test_true = evaluate(test_loader, model)

test_macro_f1 = f1_score(test_true, test_preds, average='macro', zero_division=0)
print(f"Test accuracy:  {test_acc:.3f}")
print(f"Test macro-F1:  {test_macro_f1:.3f}")
print()
print(classification_report(test_true, test_preds, target_names=class_names, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_true, test_preds)
fig, ax = plt.subplots(figsize=(max(7, n_classes), max(6, n_classes)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'PointNet++ — test set confusion matrix  (macro-F1={test_macro_f1:.2f})')
plt.tight_layout()
plt.savefig(MODELS_DIR / "test_confusion_matrix.png", dpi=120)
plt.show()

## Error analysis — hardest clusters

In [ ]:
import io
import matplotlib
matplotlib.use('Agg')

model.eval()
errors = []  # (confidence_on_true_class, true_idx, pred_idx, npz_path)
test_df_reset = test_df.reset_index(drop=True)

with torch.no_grad():
    offset = 0
    for pts, lbl in test_loader:
        pts, lbl = pts.to(device), lbl.to(device)
        probs = torch.softmax(model(pts), dim=1).cpu().numpy()
        for i, (p, l) in enumerate(zip(probs, lbl.cpu().numpy())):
            conf_true = p[l]
            pred      = p.argmax()
            if pred != l:
                global_i = offset + i
                if global_i < len(test_df_reset):
                    errors.append((float(conf_true), int(l), int(pred),
                                   test_df_reset.iloc[global_i]['npz_path']))
        offset += len(lbl)

errors.sort(key=lambda x: x[0])  # lowest confidence first
print(f"Misclassified: {len(errors)} / {len(test_ds)}")
print()
print(f"{'True class':<22} {'Predicted':<22} {'Confidence':>12}")
print('-' * 60)
for conf, true_idx, pred_idx, path in errors[:20]:
    print(f"  {class_names[true_idx]:<20}  {class_names[pred_idx]:<20}  {conf:.3f}")

In [ ]:
# Visualise the 6 hardest errors
PC_PADDING = 0.15
n_show = min(6, len(errors))

fig, axes = plt.subplots(n_show, 2, figsize=(12, 3 * n_show))
if n_show == 1: axes = [axes]

for row_i, (conf, true_idx, pred_idx, npz_path) in enumerate(errors[:n_show]):
    data  = np.load(npz_path, allow_pickle=False)
    xyz_c = data['xyz_centered'].astype(np.float32)
    rgb   = data['rgb_norm'].astype(np.float32)
    ax_t, ax_s = axes[row_i]

    for ax in (ax_t, ax_s):
        ax.set_facecolor('#1a1a1a')
        for sp in ax.spines.values(): sp.set_edgecolor('#444')
        ax.tick_params(colors='grey', labelsize=6)

    ax_t.scatter(xyz_c[:, 0], xyz_c[:, 1], c=rgb, s=8, linewidths=0)
    ax_t.set_aspect('equal')
    ax_t.set_title(f'True: {class_names[true_idx]}  |  Pred: {class_names[pred_idx]}  (conf={conf:.2f})',
                   color='salmon', fontsize=8)

    ax_s.scatter(xyz_c[:, 0], xyz_c[:, 2], c=rgb, s=8, linewidths=0)
    ax_s.set_title('Side profile', color='#888', fontsize=8)

fig.patch.set_facecolor('#1a1a1a')
plt.suptitle('Hardest misclassifications', color='white', fontsize=10)
plt.tight_layout()
plt.savefig(MODELS_DIR / "error_analysis.png", dpi=100, facecolor='#1a1a1a')
plt.show()
print(f"Saved: {MODELS_DIR / 'error_analysis.png'}")